# 03 — TRACe Judge Validation (the hard half) — Group 16

**Goal:** before we trust the LLM *judge* to score our own RAG pipeline, measure how well its TRACe scores agree with RAGBench's shipped **reference** scores — and use that to **pick which open-source judge model** to commit to. This is the **trust gate** for the entire Task-1 results matrix: every number in that matrix is only as reliable as the judge that produced it.

**How this differs from notebook 02.** Notebook 02 validated the *arithmetic* half — given gold sentence labels, our formulas reproduced the reference scores to **RMSE ~ 0**. That was a unit test. This notebook validates the *judge* half — the LLM that *produces* those labels (which sentences are relevant `R`, which were utilized `U`, whether the answer is supported).

> WARNING: **Do not expect RMSE = 0 here.** The judge makes the same *subjective* calls a human / GPT-4 annotator makes, so it will never match the reference exactly. Success is **good agreement**, not perfect reproduction — and, crucially, **the best-agreeing model among the candidates**. RAGBench's own GPT-4 annotator reached ~0.93–0.95 agreement with humans; the paper also found **relevance is the hardest metric to predict** (highest error), so expect relevance RMSE to be the worst of the three.

**What we measure** (judge vs reference, via `src/evaluator/judge_validate.py`):
- 3 fraction metrics (relevance / utilization / completeness) -> **RMSE** + mean absolute error (lower = closer).
- adherence (yes / no) -> **accuracy**.

**The method.** We feed the judge each example's *own* keyed sentences (the same `documents_sentences` / `response_sentences` GPT-4 saw), let it derive `R` / `U` / support, map those to TRACe scores with the **validated** `scores_from_label` adapter, and compare to the reference. Same inputs, compare outputs.

> Runs on **Colab GPU** (the judge is a real LLM). Budget ~20–40 min for a small run; scale `N` once you've chosen a model.

In [ ]:
# Colab setup: clone the repo and install the judge's dependencies.
import os, sys
if not os.path.exists("src"):
    !git clone https://github.com/abhisagar123/CapstoneRAG.git
    os.chdir("CapstoneRAG")
sys.path.insert(0, os.getcwd())

# datasets       = RAGBench loader
# transformers   = run the judge model        accelerate = device placement
# bitsandbytes   = 4-bit quantization (lets a 7-8B model fit a free-Colab T4)
!pip install -q datasets transformers accelerate bitsandbytes

## (optional) Hugging Face token

Only needed for the **hf/Colab** path with a **gated** model (e.g. Llama). The **local Ollama** path (default) needs no token.

**Safe setup:** add the token in Colab's **🔑 Secrets** panel (left sidebar) as `HF_TOKEN`, toggle *Notebook access* on. The cell below reads it from there — **never paste the token into a cell** (this repo is public; a committed token is a leaked credential).

In [ ]:
# (Colab only) Load HF_TOKEN from Secrets if present. Optional + safe:
#  - no token set / not on Colab -> silently continue.
# Only needed for GATED HF models (e.g. Llama on Colab). The LOCAL Ollama path
# (default below) needs no token. Never hardcode a token here: this repo is public.
try:
    from google.colab import userdata
    import os
    _tok = userdata.get('HF_TOKEN')
    if _tok:
        os.environ['HF_TOKEN'] = _tok
        print('HF_TOKEN loaded from Colab Secrets.')
    else:
        print('No HF_TOKEN secret set — fine for the local Ollama path / ungated models.')
except Exception:
    print('Not on Colab — fine; the local Ollama path needs no token.')

## 1. Choose the judge model

**Policy: run open-source models locally, but NOT Chinese models** (so no Qwen / DeepSeek / Yi / GLM).
Default path is **local Ollama** on the Mac; the `hf` (transformers/Colab) path stays available.

The judge defines ground truth, so use a strong **non-Chinese** instruct model. Options:

| Model (Ollama tag / HF id) | Params | Notes |
|---|---|---|
| `llama3.1:8b` *(default)* | 8B | Meta Llama 3.1 — strong, runs fast locally on the M3 Max |
| `mistral` / `Mistral-7B-Instruct-v0.3` | 7B | different family — good for a 2nd opinion |
| `gemma2:9b` | 9B | Google Gemma 2 — another strong non-Chinese option |
| `llama3.1:70b` | 70B | the strong judge we originally wanted — fits the M3 Max (96 GB) in 4-bit |

**Local (Ollama) prerequisite:** start the server and pull the model once:
```
ollama serve            # in a terminal (or the menu-bar app)
ollama pull llama3.1:8b
```
No HF token needed for the local path. (The `hf`/Colab path may need `HF_TOKEN` for gated models like Llama.)

This notebook validates **one (model, prompt-variant) per run** and writes its own CSV; the verdict cell
merges them. To compare another: change `MODEL` (and/or `CONSERVATIVE`), re-run — the table grows.

In [ ]:
# --- Edit these, then run top-to-bottom ---
# Policy: open-source models may run LOCALLY, but NOT Chinese models (no Qwen).
#
# JUDGE_BACKEND picks where the model runs:
#   "ollama" -> LOCAL via Ollama (recommended on the Mac; `ollama serve` + `ollama pull <model>` first)
#   "hf"     -> in-process transformers (Colab GPU; gated models need HF_TOKEN above)
JUDGE_BACKEND  = "ollama"
# MODEL — the judge under test (NON-CHINESE). Tags by backend:
#   ollama: "llama3.1:8b" (default), "mistral", "gemma2:9b", "llama3.1:70b" (M3 Max/96GB fits 4-bit)
#   hf:     "meta-llama/Llama-3.1-8B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3"
MODEL          = "llama3.1:8b"
CONSERVATIVE   = False     # True = append the "be strict" steer (targets the over-marking bias)
LOAD_IN_4BIT   = True      # (hf backend only) 4-bit on a small GPU; ignored by ollama
MAX_NEW_TOKENS = 1536      # (hf backend only) caps the OUTPUT length
N              = 50        # examples PER config (50 de-noises; 15 was too noisy)

# EXPERIMENT MATRIX (run in PARALLEL, then merge the CSVs — each writes its own file):
#   Run A: MODEL=llama3.1:8b, CONSERVATIVE=True   -> does the prompt steer cut the bias?
#   Run B: MODEL=mistral,     CONSERVATIVE=False  -> is a different model a better judge?
# NOTE: the earlier Qwen2.5-7B baseline is RETIRED (Chinese model, now disallowed) — re-baseline here.

# cuad (Legal) stays COMMENTED OUT for the hf/Colab path (long contracts OOM a T4). Locally on
# the M3 Max it MAY fit — uncomment to try. Its real TRACe scores come from notebook 04 anyway.
CONFIGS_TO_VALIDATE = [
    ("Biomedical",      "covidqa"),
    ("GenKnowledge",    "hotpotqa"),
    # ("Legal",         "cuad"),     # OOMs the hf path on a free-T4; may fit locally
    ("CustomerSupport", "emanual"),
    ("Finance",         "tatqa"),
]

In [ ]:
import src                         # registers the light components
from src.judge import load_judges
load_judges()                       # registers HuggingFaceJudge ("hf") + OllamaJudge ("ollama")

from src.registry import build
# Build via the registry. ollama = local server; hf = in-process (Colab).
if JUDGE_BACKEND == "ollama":
    judge = build("judge", "ollama", {"model": MODEL, "conservative": CONSERVATIVE})
else:
    judge = build("judge", "hf", {"model": MODEL, "load_in_4bit": LOAD_IN_4BIT,
                                  "max_new_tokens": MAX_NEW_TOKENS, "conservative": CONSERVATIVE})
print(f"Judge ready: {MODEL}  (backend={JUDGE_BACKEND}, conservative={CONSERVATIVE})")

## 2. Smoke test — does the judge emit parseable labels?

Before the (slow) full loop, run the judge on **one** labelled example. OSS models are messier than GPT-4 at clean JSON; this fails fast if the output can't be parsed, instead of wasting 20 minutes.

In [ ]:
from datasets import load_dataset
from src.evaluator.validate import REPO
from src.evaluator.judge_validate import _keyed_from_example
from src.judge import scores_from_label

ex = load_dataset(REPO, CONFIGS_TO_VALIDATE[0][1], split="test")[0]
keyed = _keyed_from_example(ex)
label = judge.label(ex["question"], keyed)        # the LLM call + JSON parse
print("Parsed judge label — keys:", sorted(label))

ours = scores_from_label(keyed, label)
print("\n              judge      reference")
for m, fld in [("relevance","relevance_score"), ("utilization","utilization_score"),
               ("completeness","completeness_score"), ("adherence","adherence_score")]:
    print(f"{m:13} {str(ours[m]):10} {ex[fld]}")
print("\nJudge produced parseable labels. (One example — agreement is measured at scale below.)")

## 3. Validate across domains

Run the judge on `N` examples of each chosen config and compare to the reference. Results append to `results/judge_validation.csv` (resumable — already-done `(model, config)` pairs are skipped, so a Colab disconnect won't lose finished configs).

In [ ]:
# Validate across the chosen configs, writing a RESUMABLE CSV. This calls the SAME
# shared helper (run_validation_sweep) that scripts/run_judge_validation.py uses, so
# the notebook and the local script can't drift. One CSV per (model, variant, N).
from src.evaluator.judge_validate import run_validation_sweep

OUT = run_validation_sweep(judge, MODEL, CONFIGS_TO_VALIDATE, n=N, conservative=CONSERVATIVE)
print("wrote:", OUT)

## 4. Verdict — agreement per model, and which judge to pick

In [ ]:
import glob
import pandas as pd

# Combine EVERY judge-validation csv in results/ (this run + any parallel runs you merged
# in). Old files predate the prompt_variant column -> default them to "baseline".
frames = []
for path in sorted(glob.glob("results/judge_validation*.csv")):
    d = pd.read_csv(path)
    if "prompt_variant" not in d.columns:
        d["prompt_variant"] = "baseline"
    frames.append(d)
df = pd.concat(frames, ignore_index=True).drop_duplicates(
    subset=["model", "prompt_variant", "config"], keep="last")

# Per (model, prompt_variant) mean agreement across the validated configs.
summary = df.groupby(["model", "prompt_variant"]).agg(
    configs           = ("config", "nunique"),
    relevance_rmse    = ("relevance_rmse", "mean"),
    relevance_signed  = ("relevance_signed", "mean"),
    utilization_rmse  = ("utilization_rmse", "mean"),
    completeness_rmse = ("completeness_rmse", "mean"),
    adherence_acc     = ("adherence_acc", "mean"),
).round(3)

print("Mean judge<->reference agreement per (model, prompt_variant):\n")
print(summary)
print("\nPer-config detail (signed +/- = DIRECTION; compare a variant to its baseline row):")
cols = ["model","prompt_variant","domain","relevance_rmse","relevance_signed",
        "utilization_rmse","completeness_rmse","adherence_acc",
        "adherence_over_flag","adherence_under_flag"]
print(df[cols].round(3).to_string(index=False))
print("\nReading it:")
print("  conservative vs baseline (same model): did relevance_signed shrink toward 0 + over_flag drop? -> steer worked.")
print("  model B vs model A (same variant): lower rmse + |signed| -> better judge.")
print("  Pick the (model, variant) with the best blend of LOW rmse and HIGH adherence_acc as THE judge.")

## 5. How to read this + honest caveats

**Picking the judge.** Lower fraction-RMSE + higher adherence accuracy = better agreement with the reference. There is **no RMSE ~ 0 pass/fail** here (unlike notebook 02) — the judge is subjective. The decision is **relative**: run 2–3 candidates, pick the best, commit to it for the whole matrix, and **report its agreement numbers** (they bound how much to trust the matrix).

- **Relevance is hardest** (RAGBench paper — highest RMSE). Don't be alarmed if it trails utilization / completeness.
- **Adherence** is a single bool from the judge's `overall_supported`; small samples make accuracy lumpy.
- **Sampled & small `N`** — this *picks* a model; it is not the final agreement report. Re-run the winner at higher `N` for the number you cite.
- **Unparseable JSON is skipped, not fatal.** OSS judges sometimes emit malformed JSON (e.g. an unescaped quote inside an explanation). The judge retries once **with sampling** (a greedy retry would repeat the same bad text), and a salvage pass repairs unescaped inner quotes. If it still won't parse, that **one example is skipped and counted** in `n_failed` — it never kills the whole config. A high `n_failed` is itself a judge-quality signal (prefer a model that parses reliably).
- **No caching yet** — re-running re-judges (cost / time). Resumability is per-`(model, config)`, not per-example; a mid-config disconnect re-does that config. (Caching is a tracked TODO.)
- **Local (Ollama)** needs no token. **hf/Colab gated models** (Llama) need license acceptance + `HF_TOKEN`. Use NON-Chinese models only.
- **`MAX_NEW_TOKENS`** may truncate the JSON on very long contexts (CUAD) -> a parse error. Raise it, or lower `N` for legal.